# v14 — is the LSTM implemented correctly, or is the task wrong for it?

Before claiming that recurrent layers hurt on this dataset, two alternative explanations have to be
ruled out. Both look plausible on inspection.

## Suspicion 1 — the selected configuration cripples its own LSTM

The selected configuration uses `fusion = 'conv'`. In the builder, that sets `seq = True`, which
forces `return_sequences=True` on the **final** LSTM layer, after which the fused sequence is reduced
by `GlobalAveragePooling1D`.

So the LSTM output is averaged across all T timesteps. But the label is the RUL at the **end** of the
window. The hidden state at t = 0 has seen one timestep; the state at t = 19 has seen the whole
window. Averaging them gives equal weight to states with almost no context, which dilutes precisely
the terminal summary the LSTM exists to produce. In the `concat` fusion path the LSTM correctly
returns only its last state — so the recurrent branch is used properly there and handicapped here.

**The ablation that found the LSTM unhelpful was run on a configuration in which the LSTM is
handicapped by construction.** That is a confound, not a result.

## Suspicion 2 — there may be no within-window temporal signal to model

Two facts about DS05, both computed below before any training:

- **RUL is constant within a flight.** All 491 flights checked have exactly one RUL value, so every
  timestep inside a window carries the same label. The label changes only *between* flights.
- **Only about 16% of sensor variance lies between flights**; the rest is within-flight variation
  driven by altitude, throttle and Mach. Not one of the 28 channels is dominated by between-flight
  variance.

A window of 20 steps at one-in-fifty sampling spans roughly 9% of a flight. Within that span the
sequence is essentially flight-condition dynamics, which carry no information about a label that does
not vary. The actual task is to estimate a *level* buried under nuisance variation — which is what an
averaging or pooling readout does well and what a recurrence-ordered model is not needed for.

If this is right, the LSTM is not badly implemented so much as **pointed at the wrong axis**: the
degradation sequence runs across flights, not within them.

## What this notebook does

| part | question | cost |
|---|---|---|
| A | Confirm the label and variance structure | free, no training |
| B | Does timestep order carry any information? Shuffle it and see. | ~25 min |
| C | Does the LSTM improve once the readout stops averaging it away? | ~35 min |
| D | Given a sequence across flights, does the LSTM earn its place? | ~30 min |

Part B is the decisive one. If shuffling the timesteps within each window does not degrade the
result, then no recurrent model can help on this representation, and the finding is about the task
rather than about LSTMs.

10 seeds per variant — enough to see effects of the size at issue, with the powered protocol of v13
held in reserve for whatever this run makes worth confirming.

In [ ]:
import os, json, gc, time, random, warnings
os.environ['PYTHONHASHSEED']='42'; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, h5py
from sklearn.metrics import mean_squared_error
from scipy import stats
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, Dropout, Dense, LSTM, Bidirectional,
                                     Concatenate, Lambda, GlobalAveragePooling1D, Rescaling)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
SEED=42; SUB=50; VAL_UNITS=[5,6]; CAP=55; EPOCHS=100; PATIENCE=15; N_SEEDS=10
SEEDS=[SEED+2000+i for i in range(N_SEEDS)]
print('TF',tf.__version__,'| GPU',bool(tf.config.list_physical_devices('GPU')),'|',N_SEEDS,'seeds')

## Part A — the label and variance structure, before any training

In [ ]:
def read_sub(d,step,chunk=400_000):
    n=d.shape[0]; out=[]
    for a in range(0,n,chunk):
        b=min(a+chunk,n); blk=d[a:b]; f=(-a)%step
        if f<len(blk): out.append(blk[f::step].astype(np.float32,copy=False))
    return np.concatenate(out)
with h5py.File('N-CMAPSS_DS05.h5','r') as h:
    Sd=read_sub(h['X_s_dev'],SUB); Vd=read_sub(h['X_v_dev'],SUB)
    Ad=read_sub(h['A_dev'],SUB);   Yd=read_sub(h['Y_dev'],SUB)
    St=read_sub(h['X_s_test'],SUB);Vt=read_sub(h['X_v_test'],SUB)
    At=read_sub(h['A_test'],SUB);  Yt=read_sub(h['Y_test'],SUB)
FD=np.hstack([Sd,Vd]); FT=np.hstack([St,Vt])
UD,CD,YD=Ad[:,0].astype(int),Ad[:,1].astype(int),Yd.ravel().astype(np.float32)
UT,CT,YT=At[:,0].astype(int),At[:,1].astype(int),Yt.ravel().astype(np.float32)
del Sd,Vd,Ad,Yd,St,Vt,At,Yt; gc.collect()
NFEAT=FD.shape[1]

kd=UD.astype(np.int64)*100000+CD
df=pd.DataFrame(FD); df['k']=kd
g=df.groupby('k')
within=g.var(ddof=0).mean(axis=0).values; between=g.mean().var(ddof=0).values
share=between/(within+between)
lab=pd.DataFrame({'k':kd,'y':YD}).groupby('k').y.nunique()
print('flights: %d | with a single RUL value: %d (%.1f%%)'%(len(lab),(lab==1).sum(),100*(lab==1).mean()))
print('median share of sensor variance lying BETWEEN flights: %.3f'%np.median(share))
print('channels where between-flight variance dominates: %d of %d'%(int((share>0.5).sum()),NFEAT))
print()
print('=> the label varies only across flights, while %.0f%% of the sensor variation'%(100*(1-np.median(share))))
print('   sits inside flights. A within-flight window therefore contains almost no')
print('   variation that the label can explain.')
del df,g; gc.collect()

## Data and model

The builder is reproduced from v12 with one change: the readout applied to the recurrent branch is
now selectable, so that `avg`, `last` and `attention` can be compared directly.

In [ ]:
TR_M=~np.isin(UD,VAL_UNITS); VA_M=np.isin(UD,VAL_UNITS)
_tr=FD[TR_M]; _mn,_mx=_tr.min(0),_tr.max(0); _rg=np.where((_mx-_mn)>1e-8,_mx-_mn,1.0)
SCALE=(1.0/_rg).astype(np.float32); OFFSET=(-_mn/_rg).astype(np.float32); del _tr; gc.collect()

def windows(F,u,c,y,mask,T):
    uu,cc,FF,yy=u[mask],c[mask],F[mask],y[mask]
    key=uu.astype(np.int64)*100000+cc; o=np.argsort(key,kind='stable')
    FF,yy,key=FF[o],yy[o],key[o]; bnd=np.flatnonzero(np.diff(key))+1
    X,Y=[],[]
    for a,b in zip(np.r_[0,bnd],np.r_[bnd,len(key)]):
        A=FF[a:b]; L=yy[a:b]
        for e in range(T,len(A)+1): X.append(A[e-T:e]); Y.append(L[e-1])
    return np.asarray(X,np.float32),np.asarray(Y,np.float32)

T=20
Xtr,ytr=windows(FD,UD,CD,YD,TR_M,T); Xva,yva=windows(FD,UD,CD,YD,VA_M,T)
Xte,yte=windows(FT,UT,CT,YT,np.ones(len(YT),bool),T)
ytr=np.minimum(ytr,CAP); yva=np.minimum(yva,CAP); yte_cap=np.minimum(yte,CAP)
TRIV=float(np.sqrt(mean_squared_error(yte_cap,np.full(len(yte_cap),ytr.mean()))))
print('T=%d | train %s | val %s | test %s | trivial %.3f'
      %(T,format(len(Xtr),','),format(len(Xva),','),format(len(Xte),','),TRIV))

BASE=dict(T=T,n_conv=2,f1=171,f_growth=1.021014565802962,kernel=7,dilation=1,
          lstm_units=57,n_lstm=1,bidir=0,fuse_filters=49,
          dropout_conv=0.20536647015555695,dropout_dense=0.36568564920359575,
          l2reg=4.080309957019978e-06,dense1=45,dense2=0,activation='relu',
          lr=0.0005334569884698839,loss='huber',huber_delta=16.334471957530912,batch=256,
          fusion='conv', rnn_readout='avg', use_lstm=1, use_cnn=1)

def last_step(x): return Lambda(lambda z: z[:,-1,:])(x)
def attn_pool(x,seed):
    sc=Dense(1,activation='tanh',kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed))(x)
    w=tf.keras.layers.Softmax(axis=1)(sc)
    return tf.keras.layers.Flatten()(tf.keras.layers.Dot(axes=1)([w,x]))
def reduce_seq(x,mode,seed):
    if mode=='avg':  return GlobalAveragePooling1D()(x)
    if mode=='last': return last_step(x)
    return attn_pool(x,seed)

def build(kw,seed):
    act=kw['activation']
    reg=regularizers.l2(kw['l2reg']) if kw['l2reg']>1e-6 else None
    kk=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),kernel_regularizer=reg)
    inp=Input(shape=(kw['T'],NFEAT)); x=Rescaling(scale=SCALE,offset=OFFSET)(inp)
    seq=(kw['fusion']=='conv'); br=[]
    if kw['use_cnn'] and kw['n_conv']>0:
        c=x
        for i in range(kw['n_conv']):
            f=max(8,int(round(kw['f1']*(kw['f_growth']**i))))
            c=Conv1D(f,kw['kernel'],padding='same',activation=act,**kk)(c)
            if kw['dropout_conv']>0: c=Dropout(kw['dropout_conv'],seed=seed)(c)
        br.append(c if seq else GlobalAveragePooling1D()(c))
    if kw['use_lstm']:
        r=x
        for j in range(kw['n_lstm']):
            rs = True if seq else (j<kw['n_lstm']-1)
            cell=LSTM(kw['lstm_units'],return_sequences=rs,**kk)
            r=Bidirectional(cell)(r) if kw['bidir'] else cell(r)
        # in sequence fusion the recurrent output stays a sequence; otherwise it is already a vector
        br.append(r)
    if not br: raise ValueError('no branch')
    if seq:
        h=br[0] if len(br)==1 else Concatenate(axis=-1)(br)
        h=Conv1D(kw['fuse_filters'],3,padding='same',activation=act,**kk)(h)
        h=reduce_seq(h,kw['rnn_readout'],seed)
    else:
        h=br[0] if len(br)==1 else Concatenate()(br)
    if kw['dense1']>=8:
        h=Dense(kw['dense1'],activation=act,**kk)(h)
        if kw['dropout_dense']>0: h=Dropout(kw['dropout_dense'],seed=seed)(h)
    m=Model(inp,Dense(1,activation='linear',**kk)(h))
    loss=tf.keras.losses.Huber(delta=kw['huber_delta']) if kw['loss']=='huber' else 'mse'
    m.compile(optimizer=tf.keras.optimizers.Adam(kw['lr']),loss=loss); return m

ROWS=[]
def run(kw,seed,Xa=None,Xv=None,Xt=None,tag=''):
    A=Xtr if Xa is None else Xa; V=Xva if Xv is None else Xv; Te=Xte if Xt is None else Xt
    K.clear_session(); gc.collect()
    os.environ['PYTHONHASHSEED']=str(seed); random.seed(seed); np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    m=build(kw,seed)
    m.fit(A,ytr,validation_data=(V,yva),epochs=EPOCHS,batch_size=kw['batch'],verbose=0,
          callbacks=[EarlyStopping('val_loss',patience=PATIENCE,restore_best_weights=True),
                     ReduceLROnPlateau('val_loss',factor=0.5,patience=7,min_lr=1e-6)])
    p=m.predict(Te,batch_size=8192,verbose=0).ravel()
    v=float(np.sqrt(mean_squared_error(yte_cap,np.clip(p,0,CAP))))
    ROWS.append(dict(tag=tag,seed=seed,capped=v)); return v

def sweep(name,kw,**kx):
    t0=time.time(); vals=[run(kw,s,tag=name,**kx) for s in SEEDS]
    print('  %-34s %.4f +/- %.4f   (%.0f min)'%(name,np.mean(vals),np.std(vals,ddof=1),(time.time()-t0)/60))
    pd.DataFrame(ROWS).to_csv('ncmapss_v14.csv',index=False)
    return np.array(vals)

## Part B — does timestep order carry any information?

The windows are shuffled along the time axis with a different permutation for every sample, applied
to training, validation and test alike. A model trained and evaluated on shuffled windows has access
to exactly the same values and none of the ordering. If the error does not rise, the ordering
contains nothing the model was using, and the case for any recurrent component collapses on this
representation regardless of how it is implemented.

In [ ]:
rs=np.random.RandomState(0)
def shuffle_time(X):
    Z=X.copy()
    for i in range(len(Z)): Z[i]=Z[i][rs.permutation(Z.shape[1])]
    return Z
Xtr_s=shuffle_time(Xtr); Xva_s=shuffle_time(Xva); Xte_s=shuffle_time(Xte)
print('Part B: ordered vs shuffled windows')
b_ord=sweep('B ordered  hybrid',dict(BASE))
b_shf=sweep('B shuffled hybrid',dict(BASE),Xa=Xtr_s,Xv=Xva_s,Xt=Xte_s)
t,p=stats.ttest_rel(b_shf,b_ord)
print()
print('  shuffling costs %+.4f RMSE (p = %.4f)'%(b_shf.mean()-b_ord.mean(),p))
print('  => %s'%('timestep order DOES carry information' if p<0.05 and b_shf.mean()>b_ord.mean()
                 else 'timestep order carries no usable information on this representation'))

## Part C — is the recurrent branch being thrown away by the readout?

Four arrangements of the same model. The first is the configuration actually selected; the rest give
the LSTM progressively fairer treatment.

In [ ]:
print('Part C: recurrent readout')
c1=sweep('C1 conv fusion + average pool (as selected)',dict(BASE))
c2=sweep('C2 conv fusion + last timestep',dict(BASE,rnn_readout='last'))
c3=sweep('C3 conv fusion + attention pool',dict(BASE,rnn_readout='attn'))
c4=sweep('C4 concat fusion, LSTM last state',dict(BASE,fusion='concat'))
c5=sweep('C5 concat fusion, LSTM only',dict(BASE,fusion='concat',use_cnn=0))
c6=sweep('C6 conv fusion, CNN only',dict(BASE,use_lstm=0))
print()
for nm,v in [('last vs avg',c2),('attention vs avg',c3),('concat vs avg',c4)]:
    t,p=stats.ttest_rel(v,c1)
    print('  %-20s %+.4f  p = %.4f'%(nm,v.mean()-c1.mean(),p))
t,p=stats.ttest_rel(c6,min([c2,c3,c4],key=lambda z:z.mean()))
print('  CNN-only vs the best LSTM arrangement: %+.4f  p = %.4f'%(c6.mean()-min(c2.mean(),c3.mean(),c4.mean()),p))

## Part D — put the sequence on the axis the label actually moves along

RUL is constant within a flight and changes between flights, so the degradation sequence runs across
flights. Each flight is reduced to a single feature vector by averaging its sampled rows, and a
sequence of `L` consecutive flights from one engine becomes one training example whose label is the
RUL of the last flight in the sequence. This is the arrangement in which a recurrent model has
something to integrate.

In [ ]:
L=12
def flight_seq(F,u,c,y,mask,L):
    uu,cc,FF,yy=u[mask],c[mask],F[mask],y[mask]
    key=uu.astype(np.int64)*100000+cc; o=np.argsort(key,kind='stable')
    FF,yy,key,uu=FF[o],yy[o],key[o],uu[o]
    bnd=np.flatnonzero(np.diff(key))+1
    rows=[];labs=[];units=[]
    for a,b in zip(np.r_[0,bnd],np.r_[bnd,len(key)]):
        rows.append(FF[a:b].mean(0)); labs.append(yy[a]); units.append(uu[a])
    rows=np.asarray(rows,np.float32); labs=np.asarray(labs,np.float32); units=np.asarray(units)
    X,Y=[],[]
    for un in np.unique(units):
        m=units==un; R=rows[m]; Lb=labs[m]
        for e in range(L,len(R)+1): X.append(R[e-L:e]); Y.append(Lb[e-1])
    return np.asarray(X,np.float32),np.asarray(Y,np.float32)

Xtr2,ytr2=flight_seq(FD,UD,CD,YD,TR_M,L); Xva2,yva2=flight_seq(FD,UD,CD,YD,VA_M,L)
Xte2,yte2=flight_seq(FT,UT,CT,YT,np.ones(len(YT),bool),L)
print('cross-flight sequences: train %s | val %s | test %s (L=%d flights)'
      %(format(len(Xtr2),','),format(len(Xva2),','),format(len(Xte2),','),L))
ytr2c=np.minimum(ytr2,CAP); yva2c=np.minimum(yva2,CAP); yte2c=np.minimum(yte2,CAP)
TRIV2=float(np.sqrt(mean_squared_error(yte2c,np.full(len(yte2c),ytr2c.mean()))))
print('trivial baseline on this representation: %.3f'%TRIV2)

def run2(kw,seed,tag):
    K.clear_session(); gc.collect()
    os.environ['PYTHONHASHSEED']=str(seed); random.seed(seed); np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    kw=dict(kw); kw['T']=L
    m=build(kw,seed)
    m.fit(Xtr2,ytr2c,validation_data=(Xva2,yva2c),epochs=EPOCHS,batch_size=64,verbose=0,
          callbacks=[EarlyStopping('val_loss',patience=PATIENCE,restore_best_weights=True),
                     ReduceLROnPlateau('val_loss',factor=0.5,patience=7,min_lr=1e-6)])
    p=m.predict(Xte2,batch_size=8192,verbose=0).ravel()
    v=float(np.sqrt(mean_squared_error(yte2c,np.clip(p,0,CAP))))
    ROWS.append(dict(tag=tag,seed=seed,capped=v)); return v

print()
print('Part D: sequence across flights')
res={}
for nm,kw in [('D1 hybrid, LSTM last state',dict(BASE,fusion='concat')),
              ('D2 LSTM only',dict(BASE,fusion='concat',use_cnn=0)),
              ('D3 CNN only',dict(BASE,fusion='concat',use_lstm=0))]:
    t0=time.time(); v=np.array([run2(kw,s,nm) for s in SEEDS]); res[nm]=v
    print('  %-30s %.4f +/- %.4f  skill %+.3f  (%.0f min)'
          %(nm,v.mean(),v.std(ddof=1),1-v.mean()/TRIV2,(time.time()-t0)/60))
    pd.DataFrame(ROWS).to_csv('ncmapss_v14.csv',index=False)
k=list(res)
t,p=stats.ttest_rel(res[k[2]],res[k[1]])
print()
print('  CNN-only vs LSTM-only on the cross-flight axis: %+.4f  p = %.4f'%(res[k[2]].mean()-res[k[1]].mean(),p))
t,p=stats.ttest_rel(res[k[1]],res[k[0]])
print('  LSTM-only vs hybrid: %+.4f  p = %.4f'%(res[k[1]].mean()-res[k[0]].mean(),p))
json.dump({k2:list(map(float,v)) for k2,v in res.items()},open('ncmapss_v14_partD.json','w'),indent=2)
pd.DataFrame(ROWS).to_csv('ncmapss_v14.csv',index=False)
print(); print('saved ncmapss_v14.csv and ncmapss_v14_partD.json')